# HeatShield AI — Colab Training

**กดรัน Cell เดียวแล้วรอ** — ได้ ZIP โมเดลใน Google Drive อัตโนมัติ

**ตั้งค่าก่อนรัน (ไม่บังคับ):**
- Colab Secrets → `CDSAPI_KEY`, `TMD_API_KEY` (ถ้าอยากใช้ ERA5/TMD)
- Runtime → Change runtime type → GPU (T4) เพื่อความเร็ว

**Output:** ZIP อยู่ที่ `MyDrive/heatshield/exports/HeatShield_artifacts_v*.zip`

In [ ]:
# ── CONFIG — แก้ตรงนี้ ──────────────────────────────────────────────────────
FAST_MODE     = True   # True = lgbm only (fast); False = lgbm+catboost ensemble
TRIALS        = 25     # Optuna trials per slot (warm-start ช่วยให้ TPE converge เร็ว)
START_DATE    = "2021-01-01"
MODEL_VERSION = "v4"   # เปลี่ยนเป็น v5, v6 ... สำหรับ run ใหม่
BACKENDS      = "lgbm" if FAST_MODE else "lgbm,catboost"
FORCE_RETRAIN = True   # True = retrain เสมอ; False = ข้าม slot ที่มีแล้ว

In [ ]:
# ============================================================
# CONFIG — แก้ตรงนี้ถ้าต้องการ
# ============================================================
FAST_MODE     = True     # True = lgbm only; False = lgbm+catboost ensemble
TRIALS        = 25       # Optuna trials per slot (warm-start ช่วยให้ TPE converge เร็ว)
START_DATE    = "2021-01-01"
MODEL_VERSION = "v4"     # เปลี่ยนเป็น v5, v6 ... สำหรับ run ใหม่
BACKENDS      = "lgbm" if FAST_MODE else "lgbm,catboost"
FORCE_RETRAIN = True     # True = retrain เสมอ; False = ข้าม slot ที่มีแล้ว
# ============================================================

import datetime, hashlib, json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path

# ── secrets ────────────────────────────────────────────────
try:
    from google.colab import drive, userdata
    for _k in ("CDSAPI_KEY", "TMD_API_KEY"):
        try: os.environ[_k] = userdata.get(_k)
        except Exception: pass
    _ON_COLAB = True
except ImportError:
    _ON_COLAB = False

# ── Drive mount ────────────────────────────────────────────
if _ON_COLAB:
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/heatshield")
else:
    DRIVE_ROOT = Path("/tmp/heatshield")

MODEL_DIR     = DRIVE_ROOT / "models" / f"forecast_{MODEL_VERSION}"
VERSIONS_DIR  = DRIVE_ROOT / "models" / "forecast_versions"
EXPORTS_DIR   = DRIVE_ROOT / "exports"
CACHE_DIR     = DRIVE_ROOT / ".cache"
DRIVE_LOG_DIR = DRIVE_ROOT / "logs" / "train"
for _d in (MODEL_DIR, VERSIONS_DIR, EXPORTS_DIR, CACHE_DIR, DRIVE_LOG_DIR):
    _d.mkdir(parents=True, exist_ok=True)

os.environ["HEATSHIELD_FORECAST_VERSION"] = MODEL_VERSION
os.environ["HEATSHIELD_DRIVE_ROOT"]       = str(DRIVE_ROOT)
os.environ["HEATSHIELD_FAST_EVAL"]        = "1"

# ── clone repo ─────────────────────────────────────────────
REPO      = "/content/Heat-wave-backend"
GH_OWNER  = "orbitorls"
GH_REPO   = "HeatShield"
GH_BRANCH = os.environ.get("HEATSHIELD_GH_BRANCH", "main")

if not Path(f"{REPO}/.git").is_dir():
    print("Cloning repo ...")
    subprocess.run(
        ["git", "clone", "--depth=1", f"--branch={GH_BRANCH}",
         f"https://github.com/{GH_OWNER}/{GH_REPO}.git", REPO],
        check=True,
    )
else:
    print("Updating repo ...")
    subprocess.run(["git", "-C", REPO, "fetch", "--all"], check=True)
    subprocess.run(["git", "-C", REPO, "reset", "--hard", f"origin/{GH_BRANCH}"], check=True)

# ── install deps ───────────────────────────────────────────
REQ       = f"{REPO}/requirements-train.txt"
HASH_FILE = CACHE_DIR / "req_hash.txt"
new_hash  = hashlib.sha256(Path(REQ).read_bytes()).hexdigest()
old_hash  = HASH_FILE.read_text().strip() if HASH_FILE.exists() else ""
if new_hash != old_hash:
    print("Installing deps ...")
    subprocess.run(["pip", "install", "-q", "-r", REQ], check=True)
    HASH_FILE.write_text(new_hash)
else:
    print("Deps unchanged — skip install")

# ── python path ────────────────────────────────────────────
os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)

# ── runtime patches ────────────────────────────────────────
def _patch_lgbm_gpu_max_bin_guard() -> None:
    """Patch older checkouts so reused Optuna GPU studies cannot crash LightGBM."""
    backend = Path(REPO) / "app" / "ml" / "forecast" / "backends" / "lgbm_backend.py"
    text = backend.read_text(encoding="utf-8")
    changed = False

    if "def _sanitize_lgbm_params_for_device" not in text:
        marker = "def _remaining_trials(study, requested_trials: int) -> int:\n"
        helper = (
            "def _sanitize_lgbm_params_for_device(params: dict, device: str) -> dict:\n"
            "    \"\"\"Return LightGBM params adjusted for known device constraints.\"\"\"\n"
            "    sanitized = dict(params)\n"
            "    if device.lower().strip() == \"gpu\" and int(sanitized.get(\"max_bin\", 0) or 0) > 255:\n"
            "        sanitized[\"max_bin\"] = 255\n"
            "    return sanitized\n\n\n"
        )
        if marker not in text:
            raise RuntimeError("Cannot patch lgbm_backend.py: _remaining_trials marker not found")
        text = text.replace(marker, helper + marker, 1)
        changed = True

    old1 = (
        "        logger.info(\"Optuna best: MAE=%.4f params=%s\", study.best_value, study.best_params)\n"
        "        self._metadata[\"n_trials_completed\"] = sum(t.state.name == \"COMPLETE\" for t in study.trials)\n"
        "        self._metadata[\"n_trials_pruned\"] = sum(t.state.name == \"PRUNED\" for t in study.trials)\n"
        "        return study.best_params\n"
    )
    new1 = (
        "        best_params = _sanitize_lgbm_params_for_device(study.best_params, dev)\n"
        "        logger.info(\"Optuna best: MAE=%.4f params=%s\", study.best_value, best_params)\n"
        "        self._metadata[\"n_trials_completed\"] = sum(t.state.name == \"COMPLETE\" for t in study.trials)\n"
        "        self._metadata[\"n_trials_pruned\"] = sum(t.state.name == \"PRUNED\" for t in study.trials)\n"
        "        return best_params\n"
    )
    if old1 in text:
        text = text.replace(old1, new1, 1)
        changed = True

    old2 = (
        "        self._metadata[\"n_trials_completed\"] = sum(t.state.name == \"COMPLETE\" for t in study.trials)\n"
        "        self._metadata[\"n_trials_pruned\"] = sum(t.state.name == \"PRUNED\" for t in study.trials)\n"
        "        return study.best_params\n"
    )
    new2 = (
        "        best_params = _sanitize_lgbm_params_for_device(study.best_params, _dev_for_name)\n"
        "        self._metadata[\"n_trials_completed\"] = sum(t.state.name == \"COMPLETE\" for t in study.trials)\n"
        "        self._metadata[\"n_trials_pruned\"] = sum(t.state.name == \"PRUNED\" for t in study.trials)\n"
        "        return best_params\n"
    )
    if old2 in text:
        text = text.replace(old2, new2, 1)
        changed = True

    if changed:
        backend.write_text(text, encoding="utf-8")
        print("Patched LightGBM GPU max_bin guard for this Colab runtime")
    else:
        print("LightGBM GPU max_bin guard already present")

_patch_lgbm_gpu_max_bin_guard()
import importlib as _importlib
_lgbm_backend = _importlib.import_module("app.ml.forecast.backends.lgbm_backend")
_sanitized_probe = _lgbm_backend._sanitize_lgbm_params_for_device({"max_bin": 511}, "gpu")
if _sanitized_probe.get("max_bin") != 255:
    raise RuntimeError(f"LightGBM GPU max_bin guard failed: {_sanitized_probe}")
print("Verified LightGBM GPU max_bin guard: 511 -> 255")

# ── patch run() into train_forecast.py if missing (older checkouts) ───────────
def _patch_train_forecast_run() -> None:
    tf_path = Path(REPO) / "scripts" / "train_forecast.py"
    tf_src  = tf_path.read_text(encoding="utf-8")
    if "def run(" in tf_src:
        return
    run_src = (
        "\n\ndef run(argv):\n"
        "    \"\"\"In-process entry point — avoids subprocess overhead on Colab.\"\"\"\n"
        "    _saved = sys.argv[:]\n"
        "    sys.argv = [\"train_forecast\"] + list(argv)\n"
        "    try:\n"
        "        main()\n"
        "    finally:\n"
        "        sys.argv = _saved\n"
    )
    tf_path.write_text(tf_src.rstrip() + run_src, encoding="utf-8")
    print("Patched train_forecast.py: added run() wrapper")

_patch_train_forecast_run()
import importlib as _il
import scripts.train_forecast as _tf_mod
_il.reload(_tf_mod)
_train_run = _tf_mod.run

# ── symlink model dir → Drive ──────────────────────────────
LOCAL_MODELS = Path(REPO) / "app" / "models" / f"forecast_{MODEL_VERSION}"
LOCAL_MODELS.parent.mkdir(parents=True, exist_ok=True)
if LOCAL_MODELS.exists() and not LOCAL_MODELS.is_symlink():
    for _item in LOCAL_MODELS.iterdir():
        _t = MODEL_DIR / _item.name
        if not _t.exists():
            shutil.move(str(_item), str(_t))
    shutil.rmtree(str(LOCAL_MODELS))
if not LOCAL_MODELS.exists():
    LOCAL_MODELS.symlink_to(str(MODEL_DIR))

# ── detect device ──────────────────────────────────────────
def _detect_device() -> str:
    try:
        result = subprocess.run(
            ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
            capture_output=True, text=True, timeout=5,
        )
        if result.returncode == 0 and result.stdout.strip():
            gpu_name = result.stdout.strip().splitlines()[0]
            print(f"GPU detected: {gpu_name} → device=gpu (auto-benchmark in lgbm_backend)")
            return "gpu"
    except Exception:
        pass
    print("No GPU detected → device=cpu")
    return "cpu"

DEVICE = _detect_device()
print(f"Device={DEVICE}")

# ── validate data ──────────────────────────────────────────
END_DATE = datetime.date.today().isoformat()
print(f"\nIngesting NASA POWER {START_DATE} → {END_DATE} ...")
subprocess.run(
    ["python", "scripts/ingest_nasa_power.py", "--start", START_DATE, "--end", END_DATE],
    cwd=REPO, check=True,
)

from datetime import date as _date
from app.data.stations import STATIONS
from app.data.loaders import read_observations

print("\nData check:")
for sid in STATIONS:
    n = len(read_observations(sid, _date.fromisoformat(START_DATE), _date.fromisoformat(END_DATE)))
    print(f"  {sid}: {n} rows")
    if n < 500:
        raise RuntimeError(f"Not enough data for {sid}: {n} rows")
print("Data OK")

# ── training helper (in-process — no subprocess overhead) ─────────────────────
LOG_DIR   = DRIVE_LOG_DIR
RUNS_ROOT = Path(REPO) / "logs" / "eval" / "runs"
LOG_DIR.mkdir(parents=True, exist_ok=True)

TIMINGS: list[dict] = []
RUN_IDS:  list[str] = []  # kept for export compatibility

def _train(tag, *, trials, horizons="6,12,24,48,72", station=None, force=False,
           model_version=MODEL_VERSION):
    argv = [
        "--trials", str(trials), "--horizons", horizons,
        "--start", START_DATE, "--end", END_DATE,
        "--backends", BACKENDS, "--model-version", model_version,
        "--device", DEVICE,
    ]
    if station:
        argv += ["--station", station]
    if force:
        argv.append("--force")
    print(f"\n{'='*60}\nTRAIN: {tag}  horizons={horizons}  station={station or 'all'}\n{'='*60}")
    t0 = time.perf_counter()
    _train_run(argv)
    elapsed = time.perf_counter() - t0
    TIMINGS.append({"tag": tag, "seconds": round(elapsed, 2)})
    print(f"\nDone in {elapsed/60:.1f} min")

# ── train all stations × all horizons (single pass) ───────────────────────────
print("\n" + "#"*60)
print("# TRAINING — all stations × all horizons")
print("#"*60)
for sid in STATIONS:
    _train(f"all_{sid}", trials=TRIALS, horizons="6,12,24,48,72",
           station=sid, force=FORCE_RETRAIN)

# ── EXPORT ─────────────────────────────────────────────────
print("\n" + "#"*60)
print("# EXPORT — building ZIP snapshot")
print("#"*60)

def _next_ver(base: Path) -> int:
    vs = [int(d.name[1:]) for d in base.iterdir()
          if d.is_dir() and d.name.startswith("v") and d.name[1:].isdigit()]
    return (max(vs) + 1) if vs else 1

slot_meta: dict = {}
warnings:  list[str] = []

# Collect finished slots from disk (in-process flow writes directly)
for _sid in STATIONS:
    for _h in [6, 12, 24, 48, 72]:
        _slot = LOCAL_MODELS / _sid / f"h{_h}"
        if (_slot / "bundle.json").exists():
            slot_meta[(_sid, _h)] = {"station": _sid, "horizon_h": _h,
                                      "backend": "lightgbm", "status": "ready"}

snap_ver = f"v{_next_ver(VERSIONS_DIR)}"
snap_dir = VERSIONS_DIR / snap_ver
snap_dir.mkdir(parents=True, exist_ok=False)

for (sid, h) in sorted(slot_meta):
    src = LOCAL_MODELS / sid / f"h{h}"
    dst = snap_dir / sid / f"h{h}"
    if src.exists():
        shutil.copytree(src, dst, dirs_exist_ok=True)
    else:
        warnings.append(f"missing:{sid}:h{h}")

cm = LOCAL_MODELS / "choice_matrix.json"
if cm.exists(): shutil.copy2(cm, snap_dir / "choice_matrix.json")

catboost_slots: list[dict] = []
for (sid, h) in sorted(slot_meta):
    cb_bundle = LOCAL_MODELS / sid / f"h{h}" / "catboost" / "bundle.json"
    if cb_bundle.exists():
        try:
            cb_meta = json.loads(cb_bundle.read_text())
            catboost_slots.append({
                "station": sid, "horizon_h": h,
                "backend": cb_meta.get("backend_name", "catboost_quantile"),
                "mae":     cb_meta.get("metrics", {}).get("mae"),
            })
        except Exception:
            pass

manifest = {
    "created_at_utc":   datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "snapshot_version": snap_ver,
    "model_version":    MODEL_VERSION,
    "backends":         BACKENDS,
    "slots":            [slot_meta[k] for k in sorted(slot_meta)],
    "catboost_slots":   catboost_slots,
    "warnings":         warnings,
    "timings":          TIMINGS,
}
(snap_dir / "manifest.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False))

stamp    = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
zip_name = f"HeatShield_artifacts_{snap_ver}_{stamp}.zip"
zip_path = EXPORTS_DIR / zip_name

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for fp in sorted(snap_dir.rglob("*")):
        if fp.is_file():
            zf.write(fp, (Path("models") / snap_ver / fp.relative_to(snap_dir)).as_posix())
    zf.writestr("manifest.json", json.dumps(manifest, indent=2, ensure_ascii=False))

mb        = zip_path.stat().st_size / 1024 / 1024
total_min = sum(t["seconds"] for t in TIMINGS) / 60

print(f"""
{'='*60}
DONE
  Snapshot   : {snap_ver}
  Model Ver  : {MODEL_VERSION}
  Backends   : {BACKENDS}
  LGBM slots : {len(slot_meta)}
  CB slots   : {len(catboost_slots)}
  ZIP        : {zip_path}
  Size       : {mb:.1f} MB
  Total      : {total_min:.0f} min
{'='*60}

ดาวน์โหลด ZIP จาก Google Drive:
  MyDrive/heatshield/exports/{zip_name}
""")

if warnings:
    print("Warnings:")
    for w in warnings:
        print(" -", w)

In [ ]:
# ── INSTALL DEPS ───────────────────────────────────────────────────────────────
import hashlib, os, shutil, subprocess, sys, time
from pathlib import Path

REPO = Path("/content/Heat-wave-backend")

# Cache deps hash in Drive if DRIVE_ROOT is available, else /tmp
_cache_dir = (DRIVE_ROOT / ".cache") if "DRIVE_ROOT" in dir() else Path("/tmp/heatshield/.cache")
_cache_dir.mkdir(parents=True, exist_ok=True)

REQ       = REPO / "requirements-train.txt"
HASH_FILE = _cache_dir / "req_hash.txt"
new_hash  = hashlib.sha256(REQ.read_bytes()).hexdigest()
old_hash  = HASH_FILE.read_text().strip() if HASH_FILE.exists() else ""
if new_hash != old_hash:
    print("Installing deps ...")
    subprocess.run(["pip", "install", "-q", "-r", str(REQ)], check=True)
    HASH_FILE.write_text(new_hash)
else:
    print("Deps unchanged — skip install")

os.chdir(str(REPO))
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

# ── DEVICE + HELPERS ──────────────────────────────────────────────────────────
import datetime

def _detect_device() -> str:
    try:
        r = subprocess.run(
            ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
            capture_output=True, text=True, timeout=5,
        )
        if r.returncode == 0 and r.stdout.strip():
            return "gpu"
    except Exception:
        pass
    return "cpu"

DEVICE   = _detect_device()
END_DATE = datetime.date.today().isoformat()

print(f"Device : {DEVICE.upper()}")
print(f"Window : {START_DATE} → {END_DATE}")

# ── Drive ↔ local model symlink ───────────────────────────────────────────────
DRIVE_MODELS = DRIVE_ROOT / "models"
DRIVE_MODELS.mkdir(parents=True, exist_ok=True)
LOCAL_MODELS  = REPO / "app" / "models" / "forecast_v3"
LOCAL_MODELS.parent.mkdir(parents=True, exist_ok=True)
if LOCAL_MODELS.is_symlink():
    pass
elif LOCAL_MODELS.exists():
    shutil.copytree(LOCAL_MODELS, DRIVE_MODELS, dirs_exist_ok=True)
    shutil.rmtree(LOCAL_MODELS)
    LOCAL_MODELS.symlink_to(DRIVE_MODELS)
else:
    LOCAL_MODELS.symlink_to(DRIVE_MODELS)

# ── Shared run state ──────────────────────────────────────────────────────────
RUNS_ROOT = DRIVE_ROOT / "runs"
RUNS_ROOT.mkdir(parents=True, exist_ok=True)
LOG_DIR   = DRIVE_ROOT / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)
TIMINGS:  list[dict] = []

# ── patch run() into train_forecast.py if missing (older checkouts) ───────────
def _patch_train_forecast_run() -> None:
    tf_path = REPO / "scripts" / "train_forecast.py"
    tf_src  = tf_path.read_text(encoding="utf-8")
    if "def run(" in tf_src:
        return
    run_src = (
        "\n\ndef run(argv):\n"
        "    \"\"\"In-process entry point — avoids subprocess overhead on Colab.\"\"\"\n"
        "    _saved = sys.argv[:]\n"
        "    sys.argv = [\"train_forecast\"] + list(argv)\n"
        "    try:\n"
        "        main()\n"
        "    finally:\n"
        "        sys.argv = _saved\n"
    )
    tf_path.write_text(tf_src.rstrip() + run_src, encoding="utf-8")
    print("Patched train_forecast.py: added run() wrapper")

_patch_train_forecast_run()
import importlib as _il
import scripts.train_forecast as _tf_mod
_il.reload(_tf_mod)
_train_run = _tf_mod.run

# ── _train() helper (in-process — no subprocess overhead) ─────────────────────
def _train(run_id: str, *, trials: int, horizons: str = "6,12,24,48,72",
           station: str | None = None, model_version: str = MODEL_VERSION,
           force: bool = False) -> None:
    argv = [
        "--horizons",      horizons,
        "--trials",        str(trials),
        "--backends",      BACKENDS,
        "--model-version", model_version,
        "--device",        DEVICE,
    ]
    if station:
        argv += ["--station", station]
    if force:
        argv.append("--force")
    print(f"\n>>> _train({run_id})  horizons={horizons}  station={station or 'all'}")
    t0 = time.time()
    _train_run(argv)
    elapsed = time.time() - t0
    TIMINGS.append({"run_id": run_id, "seconds": round(elapsed, 1)})

In [ ]:
# ── DATA: ingest NASA POWER observations ─────────────────────────────────────
print("\n" + "#"*60)
print("# DATA — ingest NASA POWER")
print("#"*60)

# Copy parquet store to /tmp to avoid Drive NFS latency during training
_drive_data = DRIVE_ROOT / "data"
_tmp_data   = Path("/tmp/heatshield_data")
if _drive_data.exists() and not _tmp_data.exists():
    print("Copying data to /tmp for faster I/O...")
    shutil.copytree(_drive_data, _tmp_data)
    _repo_data = REPO / "data"
    if _repo_data.is_symlink():
        _repo_data.unlink()
    elif _repo_data.exists():
        shutil.rmtree(_repo_data)
    _repo_data.symlink_to(_tmp_data)
    print("Data cached at /tmp.")

_r = subprocess.run(
    [sys.executable, "scripts/ingest_all.py",
     "--source", "nasa_power", "--start", START_DATE, "--end", END_DATE],
    capture_output=True, text=True,
)
if _r.returncode != 0:
    print("[warn] ingest_all.py exited", _r.returncode)
    print(_r.stderr[-2000:])
else:
    for _line in _r.stdout.splitlines():
        if any(kw in _line.lower() for kw in ("row", "station", "done", "skip", "warn")):
            print(_line)
    print("Ingest OK.")

In [ ]:
# ── TRAINING — all stations × all horizons (single pass) ─────────────────────
print("\n" + "#"*60)
print("# TRAINING — all stations × all horizons")
print("#"*60)

from app.data.stations import STATIONS

for sid in STATIONS:
    _train(f"all_{sid}", trials=TRIALS, horizons="6,12,24,48,72",
           station=sid, model_version=MODEL_VERSION, force=FORCE_RETRAIN)

total_min = sum(t["seconds"] for t in TIMINGS) / 60
print(f"\nAll stations done in {total_min:.0f} min total")

In [ ]:
# ── EXPORT: snapshot + ZIP ───────────────────────────────────────────────────
print("\n" + "#"*60)
print("# EXPORT — building ZIP snapshot")
print("#"*60)

def _next_ver(base: Path) -> int:
    vs = [int(d.name[1:]) for d in base.iterdir()
          if d.is_dir() and d.name.startswith("v") and d.name[1:].isdigit()]
    return (max(vs) + 1) if vs else 1

slot_meta: dict = {}
warnings:  list[str] = []
for rid in RUN_IDS:
    lb = RUNS_ROOT / rid / "leaderboard.json"
    if not lb.exists():
        warnings.append(f"missing_lb:{rid}")
        continue
    for r in json.loads(lb.read_text()):
        sid, h = r.get("station"), r.get("horizon_h")
        if sid in (None, "all") or h is None:
            continue
        slot_meta[(str(sid), int(h))] = {
            "station": str(sid), "horizon_h": int(h),
            "backend": r.get("backend"), "status": r.get("status"),
            "source_run_id": rid,
        }

snap_ver = f"v{_next_ver(VERSIONS_DIR)}"
snap_dir = VERSIONS_DIR / snap_ver
snap_dir.mkdir(parents=True, exist_ok=False)

for (sid, h) in sorted(slot_meta):
    src = LOCAL_MODELS / sid / f"h{h}"
    dst = snap_dir / sid / f"h{h}"
    if src.exists():
        shutil.copytree(src, dst, dirs_exist_ok=True)
    else:
        warnings.append(f"missing:{sid}:h{h}")

cm = LOCAL_MODELS / "choice_matrix.json"
if cm.exists():
    shutil.copy2(cm, snap_dir / "choice_matrix.json")

# Collect CatBoost bundle metadata
catboost_slots: list[dict] = []
for (sid, h) in sorted(slot_meta):
    cb_bundle = LOCAL_MODELS / sid / f"h{h}" / "catboost" / "bundle.json"
    if cb_bundle.exists():
        try:
            cb_meta = json.loads(cb_bundle.read_text())
            catboost_slots.append({
                "station": sid, "horizon_h": h,
                "backend": cb_meta.get("backend_name", "catboost_quantile"),
                "mae":     cb_meta.get("metrics", {}).get("mae"),
            })
        except Exception:
            pass

manifest = {
    "created_at_utc":   datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "snapshot_version": snap_ver,
    "model_version":    MODEL_VERSION,
    "backends":         BACKENDS,
    "run_ids":          RUN_IDS,
    "slots":            [slot_meta[k] for k in sorted(slot_meta)],
    "catboost_slots":   catboost_slots,
    "warnings":         warnings,
    "timings":          TIMINGS,
}
(snap_dir / "manifest.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False))

stamp    = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
zip_name = f"HeatShield_artifacts_{snap_ver}_{stamp}.zip"
zip_path = EXPORTS_DIR / zip_name

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for fp in sorted(snap_dir.rglob("*")):
        if fp.is_file():
            zf.write(fp, (Path("models") / snap_ver / fp.relative_to(snap_dir)).as_posix())
    for rid in RUN_IDS:
        run_dir = RUNS_ROOT / rid
        if run_dir.exists():
            for fp in sorted(run_dir.rglob("*")):
                if fp.is_file():
                    zf.write(fp, (Path("eval") / rid / fp.relative_to(run_dir)).as_posix())
    zf.writestr("manifest.json", json.dumps(manifest, indent=2, ensure_ascii=False))

mb        = zip_path.stat().st_size / 1024 / 1024
total_min = sum(t["seconds"] for t in TIMINGS) / 60

print(f"""
{'='*60}
DONE
  Snapshot   : {snap_ver}
  Model Ver  : {MODEL_VERSION}
  Backends   : {BACKENDS}
  LGBM slots : {len(slot_meta)}
  CB slots   : {len(catboost_slots)}
  ZIP        : {zip_path}
  Size       : {mb:.1f} MB
  Total time : {total_min:.0f} min
{'='*60}

ดาวน์โหลด ZIP จาก Google Drive:
  MyDrive/heatshield/exports/{zip_name}
""")

if warnings:
    print("Warnings:")
    for w in warnings:
        print(" -", w)

## วิธีใช้โมเดล (Windows)

```powershell
# 1. Download ZIP จาก MyDrive/heatshield/exports/

# 2. Extract + promote
$ZIP = Get-ChildItem .\HeatShield_artifacts_v*.zip | Sort LastWriteTime -Desc | Select -First 1
Expand-Archive $ZIP.FullName -DestinationPath .\artifact_unpack -Force
$VER = Get-ChildItem ".\artifact_unpack\models" -Directory | Sort Name -Desc | Select -First 1
Remove-Item app\models\forecast_v3 -Recurse -Force -ErrorAction SilentlyContinue
New-Item -ItemType Directory -Force -Path app\models\forecast_v3 | Out-Null
Copy-Item "$($VER.FullName)\*" "app\models\forecast_v3\" -Recurse -Force

# 3. Verify + evaluate
python -c "from app.ml.registry import load_latest_v3; print(load_latest_v3('BKK_01', 24).backend_name)"
python scripts/evaluate_model.py
```